# F1 Race Strategy Prediction: A Data-Driven Analysis of Modern Formula One
---
**Course Milestone Report**

| Name | Email | UID |
|------|-------|-----|
| Cooper Kerr | coopkerr@icloud.com | u1525670 |
| Isaac Middlemas | isaac.middlemas@utah.edu | u1241432 |

---

## 1. Project Description

Formula One is among the most analytically intensive motorsports in the world. Every car is outfitted with hundreds of sensors streaming data at kilohertz frequencies, and real-time strategy calls — when to pit, which tire compound to mount, whether to attempt an undercut on the car ahead — can be the difference between a podium and finishing outside the points.

This project applies data science methods to the **FastF1 API**, a publicly available Python library wrapping the official Formula 1 Timing API. Our goal is to build and evaluate a suite of predictive models that mirror tools used by real F1 strategy engineers:

1. **Race Position Prediction** — how early in a race can a driver's final finishing position be accurately predicted, and how does predictive confidence grow lap-by-lap?
2. **Tire Degradation Modeling** — fitting compound-specific degradation curves per circuit to quantify pace loss as a function of tire age.
3. **Pit Window Forecasting** — predicting how many laps remain before a viable pit window opens for a given driver, given current gap, tire state, and pace trend.

We focus on the **2022–2025 regulation era**, a coherent technical window in which car aerodynamics and tire behavior are governed by the same ruleset, making cross-season comparisons meaningful.

## 2. Data Description

### 2.1 Dataset Overview

| Property | Details |
|----------|---------|
| **Primary Source** | FastF1 Python API (wraps the official F1 Timing API and Ergast Motor Racing API) |
| **Seasons Covered** | 2020, 2021 (degradation model training); 2022, 2023, 2024, 2025 (primary analysis) |
| **Scope** | All Grand Prix race sessions; qualifying used as supplementary feature |
| **Approximate Size** | ~24 races/season × 5 seasons × 20 drivers × ~60 laps ≈ **145,000+ lap-level records** |
| **Format** | Retrieved as pandas DataFrames via Python; no static file download required |
| **Key Attributes** | Driver, team, lap number, lap time, stint number, tire compound, tire age, position, gap to leader, sector times, air temp, track temp, humidity, rainfall flag, track status (SC/VSC) |
| **Special Structures** | Time-series (laps within a race), hierarchical (laps → stints → races → seasons), geospatial (circuit layout telemetry available) |

### 2.2 Acquisition Method

All data is retrieved using the **FastF1** Python library (`pip install fastf1`). FastF1 is a well-maintained open-source package that locally caches session data after the first fetch, making repeated analysis efficient. No account or API key is required — the library accesses publicly broadcast Formula 1 timing data.

- **API Documentation:** https://docs.fastf1.dev/
- **Caching:** `fastf1.Cache.enable_cache('./f1_cache')` is enabled throughout to avoid redundant API calls during development.
- **Supplementary source:** Ergast Motor Racing API (accessed automatically through FastF1) provides historical race results used to validate final position labels.

### 2.3 Key Variables

| Variable | Type | Description |
|----------|------|-------------|
| `LapTime` | timedelta | Raw lap crossing time |
| `LapTimeSeconds` | float | Lap time converted to seconds |
| `Compound` | categorical | Tire compound: SOFT / MEDIUM / HARD / INTER / WET |
| `TyreLife` | int | Number of laps completed on the current set |
| `Position` | int | Track position at end of lap |
| `TrackStatus` | str | Composite code indicating SC (4/5), VSC (6/7), or green (1) |
| `GapAhead` / `GapBehind` | float | Computed gap (seconds) to nearest car ahead/behind |
| `threat_gap` | float | Cumulative gap to first non-pitting car behind |
| `closing_rate` | float | Pace differential between driver and threat car (s/lap) |
| `deg_delta` | float | Predicted pace loss (s) at current tire age from fitted degradation curve |
| `race_progress` | float | Lap number / total laps (0–1) |
| `sc_rate` / `vsc_rate` | float | Historical fraction of laps run under SC/VSC at this circuit |

### 2.4 Data Cleaning and Processing

**Known issues and mitigations:**

- **Outlier laps:** Laps containing pit stops, Safety Cars, Virtual Safety Cars, red flags, and formation laps are flagged and excluded from pace analysis using FastF1's `TrackStatus`, `PitInTime`, and `PitOutTime` columns.
- **Missing weather data:** Some 2022 sessions have incomplete weather telemetry; these are imputed using session-level averages or dropped depending on severity.
- **Tire age initialization:** Drivers starting on used qualifying tires have `TyreLife > 0` at lap 1. FastF1's `FreshTyre` and `TyreLife` fields are used to correct for this.
- **Lap time bounds filter:** Laps shorter than 60s or longer than 200s are dropped as physically implausible (safety car in/out laps, red flag, etc.).
- **Lap time normalization:** For cross-circuit analyses, lap times are expressed as a fuel-corrected delta from each driver's session baseline rather than absolute seconds.

### 2.5 Derived Features / Feature Engineering

| Feature | Description |
|---------|-------------|
| `stint_relative_lap` | Lap number within current stint (resets at each pit stop) |
| `gap_trend` | Linear slope of threat gap over the last 3 laps — indicates whether a window is opening or closing |
| `rolling_avg_pace` | 5-lap rolling mean of lap time (smoothed pace) |
| `deg_delta` | Quadratic degradation curve output at current `TyreLife` |
| `fuel_correction` | `(lap_number - 1) × 1.8 kg/lap × 0.035 s/kg` — corrects for car weight reducing as fuel burns |
| `pit_loss_fraction` | Circuit pit loss time divided by median lap time — normalises pit stop cost across circuits |
| `cars_behind_pitting` | Count of cars behind that pit within the next 3 laps — strategic pressure indicator |

## 3. Ethical Data Concerns

### 3.1 Stakeholder Analysis

| Stakeholder Group | How They Are Affected | Potential Harms |
|-------------------|-----------------------|-----------------|
| **F1 Drivers** | Performance data is analyzed at the individual level — lap times, stint choices, and race pace are attributed to named individuals | Could unfairly reinforce narratives about a driver's performance (e.g., labeling a driver as "slow in high-temperature conditions") based on limited or confounded data |
| **F1 Teams** | Strategy patterns and tactical tendencies are made transparent and potentially predictable | Detailed pit window models derived from public data could theoretically be used to anticipate or counter a team's strategy, though all source data is already publicly broadcast |
| **Broadcasters and Betting Markets** | Predictive race outcome models could inform real-time commentary tools or be adapted for sports betting contexts | If a position-prediction model were productized for betting, it could contribute to gambling-related harms |
| **General Public / Fans** | Greater transparency into how race outcomes are shaped by strategy rather than pure driver skill | Could diminish perceived sporting merit if outcomes appear heavily predetermined by models |

### 3.2 Most Significant Risk: Misattribution of Performance

The most meaningful ethical risk is **misattribution of individual driver performance**. Lap time and race position are shaped by many factors outside a driver's direct control — car performance tier, team strategy calls, mechanical reliability, and weather. A naive model that characterizes a driver as underperforming in certain conditions, without controlling for car-level confounders, could unfairly damage a professional athlete's reputation.

**Mitigation strategies:**
- All individual performance analyses explicitly control for car/team tier as a covariate.
- Findings are framed at the **strategic and systemic level** (e.g., "SOFT tires degrade 0.04s/lap faster at Bahrain in temperatures above 38°C") rather than as individual driver critiques.
- We do not publish raw driver-level performance rankings without substantial caveats about confounding factors.

### 3.3 Privacy Considerations

All data used in this project is publicly broadcast by Formula One Management during live race weekends and is subsequently accessible through official APIs. There is no personally identifiable information (PII) beyond the names of professional athletes who operate as public figures. No private, proprietary, or unpublished team data will be used at any point.

### 3.4 Scope and Intended Use

While our data is fully public, combining multi-season telemetry into predictive strategy tools goes somewhat beyond casual fan use. We are transparent in this write-up about the academic scope and limitations of our models, and we do not intend to make them available for commercial or betting applications.

## 4. Methods

Our project is organized into three interconnected modeling pipelines, each targeting a distinct research question. All pipelines share a common data loading and feature engineering layer.

---

### 4.1 Race Position Prediction (Random Forest Classifier)

**Research question:** At what point in a race (lap N) can a driver's final finishing position be predicted with meaningful accuracy? How does predictive confidence evolve as the race progresses?

**Target variable:** Final finishing position grouped into four buckets to reduce sensitivity to last-lap anomalies:
- Podium (P1–P3), Top Points (P4–P6), Low Points (P7–P10), Non-Points (P11+)

**Features at lap N:** Current track position, tire compound (one-hot encoded), tire age, stint number, and lap number. In the extended version, gap-to-leader and rolling pace trend are added.

**Model:** `RandomForestClassifier` (scikit-learn, 100 estimators, `random_state=42`). Random Forest was selected for its ability to capture nonlinear thresholds (e.g., the sharp tire performance cliff) and its native feature importance output.

**Evaluation:** A "predictability curve" — classification accuracy measured at lap checkpoints (5, 10, 20, 30, 40, 50) — is constructed across multiple races for both the 2024 and 2025 seasons, allowing year-over-year competitiveness comparison.

**Train/test split:** Models are trained on all available lap observations from a given race set, and accuracy is evaluated within the same dataset at fixed lap snapshots. Future work will use a proper temporal holdout.

---

### 4.2 Pit Window Forecasting (Gradient Boosted Regressor)

**Research question:** Given the current race state (tire age, inter-car gaps, pace differential, circuit characteristics), how many laps remain until a viable pit window opens for a given driver?

**Target variable:** `laps_until_open` — the number of laps until the threat gap behind the driver (gap to the first non-pitting car) exceeds `pit_loss + clean_air_buffer (3.0s)`. Rows where no window opens within a 20-lap horizon are assigned a sentinel value of 21 ("window does not open soon"), keeping them as informative negative examples.

**Key features:**

| Feature | Rationale |
|---------|-----------|
| `threat_gap` | Primary signal: gap to first non-pitting car behind |
| `closing_rate` | Pace differential — a fast car 8s back closes faster than a slow one |
| `gap_trend` | 3-lap slope of threat gap — indicates trajectory |
| `tire_age` / `deg_delta` | Current tire state and projected degradation |
| `pit_loss` / `pit_loss_fraction` | Circuit-specific pit stop cost |
| `sc_rate` / `vsc_rate` | Historical safety car frequency at this circuit |
| `race_progress` | Position in the race affects pit strategy options |
| `context` | Gap state: tight_ahead / tight_behind / tight_both / free |
| `compound` | Compound affects degradation trajectory and urgency |

**Window viability check (pace-adjusted):** Rather than a static gap threshold, we use a dynamic check that accounts for relative pace:
- If the threat car is faster, we compute whether they can close the gap before the pitting driver rejoins with clean air.
- This prevents the model from recommending pits when a much faster car would pass regardless.

**Model:** `GradientBoostingRegressor` (scikit-learn, 300 estimators, learning rate 0.05, max depth 5, subsample 0.8). Hyperparameters are tuned via grid search on a 2022 validation split with 2023 training data, then evaluated on a **2024 temporal holdout**.

**Evaluation:** MAE, RMSE, "within 2 laps" accuracy, "within 5 laps" accuracy, and per-context MAE breakdown.

---

### 4.3 Libraries and Tools

| Library | Role |
|---------|------|
| `fastf1` | Data acquisition and session loading |
| `pandas` / `numpy` | Tabular processing, feature engineering |
| `scikit-learn` | Random Forest, Gradient Boosting, preprocessing, evaluation |
| `statsmodels` | Regression with confidence intervals (temperature analysis) |
| `matplotlib` / `seaborn` | Static visualization |
| `plotly` | Interactive lap-by-lap position animations |
| `joblib` | Model serialization |

# 5. Preliminary Results

This section presents the preliminary code and outputs from our two main modeling pipelines. The data loading infrastructure, degradation model, and pit window feature engineering pipelines are all functional and validated. The race position classifier has been run against 2024 data and produces sensible predictability curves.

## 5.1 Race Position Prediction: 2024 vs 2025 Analysis

## Objective
To determine at what point in a Formula 1 race a driver's final finishing position can be predicted with high accuracy, and how this predictive confidence evolves as the race progresses. By incorporating both the 2024 and 2025 seasons into the analysis, we can comparative evaluate whether the racing product has become more or less predictable year-over-year as teams converge in their car development under the current technical regulations.

## Methodology

### 1. Data Acquisition
We will use the **FastF1** Python API to cleanly extract lap-by-lap timing and context for races across the seasons:
- **2024 Dataset:** Comprehensive set of Grand Prix sessions.
- **2025 Dataset:** Comparative set of Grand Prix sessions (e.g. comparing the same tracks run in 2024).

### 2. Feature Engineering (Lap $N$ State)
To predict a driver's final position, we freeze the state of the race at lap $N$. For each driver, we capture:
- **Current Position:** Track position at the end of lap $N$.
- **Gap to Leader / Gap Ahead:** Immediate time deltas indicating density of the field.
- **Tyre Compound & Age:** Current tyre rubber (Soft/Medium/Hard) and the number of laps it has done.
- **Pit Stops Taken:** Number of pit stops completed by lap $N$.
- **Pace Trend:** Rolling average of the driver's lap times relative to the field over the last 3-5 laps.

### 3. Target Variable
We will predict the final finishing position grouped into buckets to smooth out minor anomalies (e.g., last lap DNF or strategic point-hunting pits):
- **Buckets:** Podium (P1-P3), Top Points (P4-P6), Low Points (P7-P10), Non-Points (P11+).

### 4. Modeling Approach
- **Algorithm:** **Random Forest Classifier**. Selected because it natively handles non-linear variables (like the sharp performance cliff of aging tyres) and easily maps feature importance.
- **Evaluation Loop:** 
  1. Train models independently using the isolated state variables at specific lap intervals (e.g., Laps 5, 15, 30, 45).
  2. Compute evaluation metrics (Accuracy, Precision, Recall) on a hold-out test set at each interval.
  3. Visualize as a **"Predictability Curve"**.

---

### Data Pulling and Feature Extraction

In [ ]:
import fastf1
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

fastf1.Cache.enable_cache('fastf1_cache')  # Cache is strictly necessary for processing multiple races

def extract_features_at_lap(year, grand_prix, target_lap):
    """
    Fetches the race session for a specific year and extracts the race state 
    for all drivers specifically at the target_lap.
    """
    try:
        session = fastf1.get_session(year, grand_prix, 'R')
        session.load(telemetry=False, weather=False, messages=False)
    except Exception as e:
        print(f"Failed to load {year} {grand_prix}: {e}")
        return None
        
    laps = session.laps
    if laps.empty:
        return None

    # Filter laps ending at the target
    current_state = laps[laps['LapNumber'] == target_lap].copy()
    if current_state.empty:
        return None
    
    # Target variable lookup: Final race position
    final_positions = []
    for driver in current_state['DriverNumber']:
        driver_laps = laps[laps['DriverNumber'] == driver]
        if not driver_laps.empty:
            final_pos = driver_laps.iloc[-1]['Position']
            final_positions.append(final_pos)
        else:
            final_positions.append(np.nan)
            
    current_state['FinalPosition'] = final_positions
    current_state['Year'] = year
    current_state['GrandPrix'] = grand_prix
    
    # Extract only the features we need
    features = current_state[['Year', 'GrandPrix', 'DriverNumber', 'Position', 'Compound', 'TyreLife', 'FinalPosition']]
    
    return features.dropna()

### Training the Classifier

In [ ]:
def train_and_eval_for_lap(df_features):
    """
    Trains a Random Forest classifier to predict final position bucket and returns validation accuracy.
    """
    # Create target position buckets
    bins = [0, 3, 6, 10, 25]
    labels = ['Podium', 'Top Points', 'Low Points', 'Out of Points']
    df_features['PosBucket'] = pd.cut(df_features['FinalPosition'], bins=bins, labels=labels)
    
    # One-hot encode categorical features (like Tyre Compound)
    df_encoded = pd.get_dummies(df_features, columns=['Compound'], drop_first=True)
    
    # Establish X and y
    feature_cols = ['Position', 'TyreLife'] + [col for col in df_encoded.columns if 'Compound_' in col]
    X = df_encoded[feature_cols]
    y = df_encoded['PosBucket']
    
    if len(X) < 10:
        return np.nan # Not enough data
        
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    preds = rf.predict(X_test)
    return accuracy_score(y_test, preds)

### Predictability Curve (2024 vs 2025)

In [ ]:
def run_comparison_analysis(races_list):
    """
    Iterates through specific lap checkpoints across a 2024 vs 2025 race set to map predictability growth.
    """
    lap_checkpoints = [5, 10, 20, 30, 40, 50]
    results_2024 = []
    results_2025 = []
    
    for lap in lap_checkpoints:
        # Build dataset for 2024 checkpoint
        races_24 = [extract_features_at_lap(2024, gp, lap) for gp in races_list]
        df_2024 = pd.concat([df for df in races_24 if df is not None], ignore_index=True) if races_24 else pd.DataFrame()
        
        # Build dataset for 2025 checkpoint
        races_25 = [extract_features_at_lap(2025, gp, lap) for gp in races_list]
        df_2025 = pd.concat([df for df in races_25 if df is not None], ignore_index=True) if races_25 else pd.DataFrame()
        
        # Evaluate predictability
        if not df_2024.empty:
            results_2024.append(train_and_eval_for_lap(df_2024))
        else:
            results_2024.append(np.nan)
            
        if not df_2025.empty:
            results_2025.append(train_and_eval_for_lap(df_2025))
        else:
            results_2025.append(np.nan)
            
    # Visualization: The Predictability Curve Comparison
    plt.figure(figsize=(10, 6))
    plt.plot(lap_checkpoints, results_2024, marker='o', linewidth=2, label='2024 Season Accuracy')
    plt.plot(lap_checkpoints, results_2025, marker='s', linewidth=2, label='2025 Season Accuracy', color='darkorange')
    
    plt.title('F1 Race Outcome Predictability Evolution: 2024 vs 2025', fontsize=14)
    plt.xlabel('Race Lap Number', fontsize=12)
    plt.ylabel('Prediction Accuracy (Position Bucket)', fontsize=12)
    plt.ylim(0, 1.05)
    plt.axhline(0.25, color='red', linestyle='--', label='Random Guess Baseline')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.show()

# --- Execution Entry Point ---
# Choose a subset of comparable early-to-mid season races
target_races = ['Bahrain', 'Saudi Arabia', 'Australia', 'Japan', 'Miami']
run_comparison_analysis(target_races)

## Expected Analytical Outcomes
1. **The Predictability Curve Profile:** Accuracy is heavily dependent on the lap metric. At Lap 5, dense traffic and unstratified pit-stop strategies expect accuracy down near ~40%. By lap 45, true pace dictates running position, and predictable outcomes are likely to rise above 80%-90% accuracy.
2. **2024 vs 2025 Insights:** Plotting these trendlines over each other creates immediate context for analyzing dominance shifts. If 2024 (e.g., highly controlled early runs by distinct dominant teams) slopes up sharper and earlier than 2025 (indicating perhaps a closer, unpredictable field of converged aerodynamic concepts), it tells a structural story about the competitiveness and entertainment value of the sport as the regulations age.

---

## 5.2 Pit Window Forecasting — Code Notebook: Data, Methodology & Expected Outcomes

This section provides a structured narrative overview of the pit window forecasting code notebook (`F_p2.ipynb`), covering the data it operates on, the modeling approach, the key variables involved, and the analytical outcomes we expect to observe.

---

### Data Description

The pit window forecasting notebook operates on lap-level race data retrieved from the **FastF1 API** across the **2020–2024 seasons**. The dataset is split into two roles:

- **Degradation training years (2020–2021):** Used exclusively to fit the tire degradation curves. These seasons are kept separate to prevent data leakage — the degradation model is a feature input to the pit window model, so it must be trained on data the pit window model has never seen.
- **Label years (2022–2024):** Used to construct the forward-looking pit window labels and train the gradient boosted regressor. The 2024 season serves as the temporal holdout test set.

The notebook covers **27 circuits** with explicitly profiled pit loss times (ranging from 20.8s at the Austrian Grand Prix to 24.5s at the Mexican Grand Prix), reflecting real-world variation in pit lane length and speed limit differences across venues.

**Data filtering applied:**
- Laps shorter than 60s or longer than 200s are dropped (covers in/out laps, red flag laps, anomalous timing)
- Safety Car, Virtual Safety Car, and yellow flag laps are excluded from degradation fitting using composite `TrackStatus` codes (`{'2','4','5','6','7'}`)
- Opening lap and the final 15% of the race are excluded from label construction (pit stops in these windows are not strategically informative)
- Pit entry and exit laps are excluded from driver observations (the car is not in a "decide whether to pit" state)

---

### Methodology

The notebook is organized into six sequential phases, each building on the previous:

#### Phase 1 — Data Loading
A standardized loader (`load_race_laps`) fetches any race by year and round number, applies the base quality filter, and returns two DataFrames: `clean_laps` (green-flag only, used for degradation fitting) and `all_laps` (all valid laps, used for gap computation and label construction). A `check_pipeline()` function validates lap counts, gap distribution, and circuit profile coverage before any modeling begins.

#### Phase 2 — Gap Computation & Context Classification
Inter-car gaps are computed from **lap crossing times** (the `Time` column in FastF1, representing elapsed session time when each car completes a lap). For each lap, cars are sorted by position and the time difference between adjacent crossings gives `GapAhead` and `GapBehind`.

A critical subtlety is handled here: **lapped cars are excluded** from gap computation by filtering to only cars on the same lap count as the driver of interest (`same_lap` filter). Without this, a driver who has lapped a backmarker would appear to have a very small gap to the car "ahead," when in reality they are racing a different car entirely.

Each driver's gap state is then classified into one of four **racing contexts**:
| Context | Meaning |
|---------|---------|
| `tight_ahead` | Car ahead is within 5s — active reference |
| `tight_behind` | Car behind is within 5s — undercut threat |
| `tight_both` | Racing references on both sides |
| `free` | No immediate racing references |

#### Phase 3 — Tire Degradation Model
A **quadratic polynomial** is fit to fuel-corrected pace deltas (observed lap time minus driver's minimum corrected lap time) as a function of `TyreLife`, separately for each circuit-compound combination. The fuel correction removes the confounding effect of decreasing car weight as fuel burns:

```
fuel_correction(lap) = (lap − 1) × 1.8 kg/lap × 0.035 s/kg
```

Degradation outliers (pace delta > 5s) are trimmed before fitting. The fitted curves are evaluated on held-out 2023–2024 data using RMSE and MAE per circuit-compound pair. **Compound crossover laps** — the stint length at which a harder compound becomes faster in cumulative time — are also computed per circuit and serve as a strategic reference for optimal pit timing.

#### Phase 4 — Forward-Looking Label Construction
This is the core engineering step. For every driver on every eligible lap, the notebook scans forward up to **20 laps** and identifies the first lap at which a viable pit window opens. The window viability check is **pace-adjusted** rather than static:

- A **static check** simply asks: is `threat_gap > pit_loss + 3.0s`?
- The **dynamic check** accounts for relative pace: if the threat car is faster, it computes whether they would close the gap before the pitting driver rejoins with clean air.

This distinction matters significantly. A Red Bull 8s behind a Williams will close that gap in a few laps; the window is not truly open. A Williams 8s behind a Red Bull will never close it; the window is wide open regardless of the static threshold.

The resulting target variable `laps_until_open` represents the number of laps until the window opens. Rows where no window opens within the 20-lap horizon are assigned a **sentinel value of 21** ("window does not open soon"), keeping them in the training set as informative negative examples rather than being discarded.

#### Phase 5 — Full Dataset Construction
`build_full_dataset()` orchestrates Phases 1–4 across all label years, concatenating the per-race label DataFrames into a single dataset serialized to `f1_pit_window_labels.csv`. Expected output: **~145,000 driver-lap observations** across 2022–2024, covering 27 circuits and all 20 drivers per race.

#### Phase 6 — Model Training & Hyperparameter Tuning
A **Gradient Boosted Regressor** (`sklearn.ensemble.GradientBoostingRegressor`) is trained on 2022–2023 data and evaluated on a **2024 temporal holdout**. This mirrors real deployment: the model always predicts future races from past data, never the reverse.

Hyperparameter tuning is performed via grid search over:
- `n_estimators`: [200, 300, 500]
- `max_depth`: [4, 5, 6]
- `learning_rate`: [0.03, 0.05, 0.1]
- `min_samples_leaf`: [10, 20, 30]

The search uses 2022 as validation, trains on 2023, then refits the best parameters on the full 2022–2023 training set before evaluating on 2024.

---

### Key Variables

| Variable | Role | Description |
|----------|------|-------------|
| `threat_gap` | Primary feature | Cumulative gap (s) to first non-pitting car behind |
| `closing_rate` | Feature | `own_pace − threat_pace` (s/lap); positive = threat car faster |
| `gap_trend` | Feature | Linear slope of threat gap over last 3 laps |
| `tire_age` | Feature | Laps completed on current tire set |
| `deg_delta` | Feature | Predicted pace loss (s) at current tire age from fitted curve |
| `stint_number` | Feature | Which pit stop cycle the driver is on |
| `pit_loss` | Feature | Circuit-specific pit stop time cost (seconds) |
| `pit_loss_fraction` | Feature | `pit_loss / median_lap_time` — normalised across circuits |
| `circuit_median_gap` | Feature | Median inter-car gap at this circuit — encodes how processional vs. variable the racing tends to be |
| `sc_rate` / `vsc_rate` | Feature | Historical fraction of laps run under Safety Car / Virtual Safety Car — encodes likelihood of a free pit stop opportunity |
| `race_progress` | Feature | `lap / total_laps` — position in race affects strategic options |
| `context` | Feature (categorical) | Gap state: tight_ahead / tight_behind / tight_both / free |
| `compound` | Feature (categorical) | Current tire compound — affects degradation trajectory |
| `own_pace` / `threat_pace` | Feature | Recent 5-lap rolling average pace for driver and threat car |
| `laps_until_open` | **Target** | Laps until pit window opens (0–20); 21 = does not open in horizon |
| `gap_is_open_now` | Secondary target | Binary: is the window currently open? |
| `opens_in_horizon` | Secondary target | Binary: does the window open within 20 laps? |

---

### Constants and Data Loader

In [ ]:
import fastf1
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

fastf1.Cache.enable_cache('./f1_cache')

# ─────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────

FUEL_BURN_RATE  = 1.8    # kg per lap
FUEL_LAP_EFFECT = 0.035  # seconds per kg of fuel
CLEAN_AIR_BUFFER = 3.0   # seconds of clear air required post-pit
FORECAST_HORIZON = 20    # laps forward to scan for window opening

CIRCUIT_PROFILES = {
    'Bahrain Grand Prix':        {'pit_loss': 21.5},
    'Saudi Arabian Grand Prix':  {'pit_loss': 24.1},
    'Australian Grand Prix':     {'pit_loss': 23.8},
    'Italian Grand Prix':        {'pit_loss': 21.1},
    'British Grand Prix':        {'pit_loss': 22.4},
    'Hungarian Grand Prix':      {'pit_loss': 21.8},
    'Belgian Grand Prix':        {'pit_loss': 24.0},
    'Spanish Grand Prix':        {'pit_loss': 22.0},
    'Azerbaijan Grand Prix':     {'pit_loss': 23.5},
    'Austrian Grand Prix':       {'pit_loss': 20.8},
    'Dutch Grand Prix':          {'pit_loss': 23.0},
    'United States Grand Prix':  {'pit_loss': 23.2},
    'Mexico City Grand Prix':    {'pit_loss': 24.5},
    'São Paulo Grand Prix':      {'pit_loss': 23.8},
    'Singapore Grand Prix':      {'pit_loss': 23.1},
    'Japanese Grand Prix':       {'pit_loss': 22.6},
    'Abu Dhabi Grand Prix':      {'pit_loss': 23.0},
    'Emilia Romagna Grand Prix': {'pit_loss': 22.0},
    'Monaco Grand Prix':         {'pit_loss': 22.0},
    'French Grand Prix':         {'pit_loss': 22.5},
    'Styrian Grand Prix':        {'pit_loss': 20.8},
    '70th Anniversary Grand Prix': {'pit_loss': 22.4},
    'Eifel Grand Prix':          {'pit_loss': 22.0},
    'Turkish Grand Prix':        {'pit_loss': 23.0},
    'Miami Grand Prix':          {'pit_loss': 23.3},
    'Las Vegas Grand Prix':      {'pit_loss': 24.0},
    'Qatar Grand Prix':          {'pit_loss': 23.5},
    'Canadian Grand Prix':       {'pit_loss': 22.8},
}


# ─────────────────────────────────────────────
# Track status filter
# ─────────────────────────────────────────────

def is_clean_lap(status: str) -> bool:
    """
    Returns True only for pure green flag laps.
    TrackStatus is a composite string — '124' means
    yellow + safety car active simultaneously.
    Any non-green code disqualifies the lap.
    """
    if pd.isna(status):
        return False
    return not any(c in {'2', '4', '5', '6', '7'} for c in str(status))


# ─────────────────────────────────────────────
# Gap computation
# ─────────────────────────────────────────────

def compute_inter_car_gaps(laps: pd.DataFrame) -> pd.DataFrame:
    """
    Compute gap to car directly ahead and directly behind
    for every driver on every lap, using the session Time
    column (elapsed seconds when each car crosses the line).

    gap = Time[car_behind] - Time[car_ahead]

    Cars are grouped by (Circuit, Year, LapNumber) and
    sorted by Position. The gap between adjacent positions
    is the difference in their crossing times.
    """
    laps = laps.copy()
    laps['TimeSeconds'] = laps['Time'].dt.total_seconds()
    gap_records = []

    for (circuit, year, lap_num), group in laps.groupby(
        ['Circuit', 'Year', 'LapNumber']
    ):
        group = group.dropna(subset=['Position', 'TimeSeconds'])
        group = group.sort_values('Position')

        times   = group['TimeSeconds'].values
        drivers = group['Driver'].values

        for i in range(len(drivers)):
            gap_ahead  = times[i] - times[i-1] if i > 0 else np.nan
            gap_behind = times[i+1] - times[i] if i < len(times)-1 else np.nan

            gap_records.append({
                'Driver':    drivers[i],
                'LapNumber': lap_num,
                'Circuit':   circuit,
                'Year':      year,
                'GapAhead':  round(gap_ahead,  3) if not np.isnan(gap_ahead)  else np.nan,
                'GapBehind': round(gap_behind, 3) if not np.isnan(gap_behind) else np.nan,
            })

    return pd.DataFrame(gap_records)


# ─────────────────────────────────────────────
# Data loader
# ─────────────────────────────────────────────

def load_race_laps(
    year: int,
    round_number: int
) -> tuple[pd.DataFrame, pd.DataFrame] | tuple[None, None]:
    """
    Load a single race. Returns two dataframes:

    clean_laps — SC/VSC/yellow laps removed.
                 Used for degradation model fitting only.

    all_laps   — base quality filter only, pit entry laps
                 retained. Used for gap computation and
                 forward-looking label construction.
    """
    try:
        session = fastf1.get_session(year, round_number, 'R')
        session.load(telemetry=False, weather=False, messages=False)
        laps = session.laps.copy()

        laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
        laps['Circuit']        = session.event['EventName']
        laps['Year']           = year

        base = laps[
            laps['LapTime'].notna() &
            laps['Compound'].notna() &
            laps['TyreLife'].notna() &
            laps['LapTimeSeconds'].notna() &
            (laps['LapNumber'] > 1) &
            (laps['LapTimeSeconds'] > 60) &
            (laps['LapTimeSeconds'] < 200)
        ].copy()

        all_laps   = base.copy()
        clean_laps = base[base['TrackStatus'].apply(is_clean_lap)].copy()

        print(f"  {year} R{round_number} {session.event['EventName']}: "
              f"{len(clean_laps)} clean / {len(all_laps)} total laps")
        return clean_laps, all_laps

    except Exception as e:
        print(f"  Skipped {year} R{round_number}: {e}")
        return None, None


def get_rounds_for_circuits(year: int, target_circuits: list[str]) -> list[int]:
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    return [
        int(event['RoundNumber'])
        for _, event in schedule.iterrows()
        if event['EventName'] in target_circuits
    ]


def load_dataset(years: list[int]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load all races from the given years that appear in
    CIRCUIT_PROFILES. Returns (clean_laps, all_laps).
    """
    clean_frames, all_frames = [], []
    target_circuits = list(CIRCUIT_PROFILES.keys())

    for year in years:
        rounds = get_rounds_for_circuits(year, target_circuits)
        if not rounds:
            print(f"  No matching circuits for {year}")
            continue
        print(f"\n  {year}: rounds {rounds}")
        for rnd in rounds:
            clean, all_ = load_race_laps(year, rnd)
            if clean is not None and len(clean) > 0:
                clean_frames.append(clean)
            if all_ is not None and len(all_) > 0:
                all_frames.append(all_)

    if not clean_frames and not all_frames:
        raise ValueError("No data loaded.")

    clean_df = pd.concat(clean_frames, ignore_index=True) if clean_frames else pd.DataFrame()
    all_df   = pd.concat(all_frames,   ignore_index=True) if all_frames   else pd.DataFrame()
    return clean_df, all_df


# ─────────────────────────────────────────────
# Pipeline check
# ─────────────────────────────────────────────

def check_pipeline(year: int = 2023, round_number: int = 2) -> None:
    """
    Load one race and verify the pipeline is working.
    Checks lap counts, gap distribution, and pit stop count.
    """
    clean, all_laps = load_race_laps(year, round_number)
    if all_laps is None:
        print("FAILED — could not load session")
        return

    gaps = compute_inter_car_gaps(all_laps)
    valid_gaps = gaps['GapAhead'].dropna()
    valid_gaps = valid_gaps[(valid_gaps > 0) & (valid_gaps < 120)]

    pit_stops = all_laps['PitOutTime'].notna().sum()
    circuit   = all_laps['Circuit'].iloc[0]

    print(f"\nPipeline check — {year} R{round_number} {circuit}")
    print(f"  Clean laps    : {len(clean)}")
    print(f"  Total laps    : {len(all_laps)}")
    print(f"  Pit stops     : {pit_stops}")
    print(f"  Gap median    : {valid_gaps.median():.2f}s")
    print(f"  Gap p99       : {valid_gaps.quantile(0.99):.2f}s")
    print(f"  Gap % > 25s   : {(valid_gaps > 25).mean():.1%}")
    print(f"  Circuit in profiles: {circuit in CIRCUIT_PROFILES}")
    print("\nPipeline OK" if circuit in CIRCUIT_PROFILES else "\nWARNING: circuit not in profiles")

### Phase 2 — Context classification

In [ ]:
def get_racing_gaps(
    lap_slice: pd.DataFrame,
    driver: str,
) -> tuple[float, float]:
    """
    Returns (gap_ahead, gap_behind) to the nearest car on the
    SAME LAP as this driver — excluding lapping leaders or
    cars being lapped.

    FastF1 assigns each car its own LapNumber based on how many
    laps it has personally completed. A lapped car will have a
    lower LapNumber than the leader at the same point in real
    time. Filtering by driver_lap_num ensures we only measure
    gaps to cars that are actually racing this driver.
    """
    driver_row = lap_slice[lap_slice['Driver'] == driver]
    if len(driver_row) == 0:
        return np.nan, np.nan

    driver_lap_num = driver_row.iloc[0]['LapNumber']
    driver_pos     = driver_row.iloc[0]['Position']

    # Only cars on the same lap count
    same_lap = lap_slice[
        lap_slice['LapNumber'] == driver_lap_num
    ].sort_values('Position')

    # Gap ahead — nearest same-lap car with lower position number
    cars_ahead = same_lap[same_lap['Position'] < driver_pos].sort_values(
        'Position', ascending=False
    )
    gap_ahead = np.nan
    if len(cars_ahead) > 0:
        # GapBehind of the car directly ahead = gap between them and driver
        ahead_pos = cars_ahead.iloc[0]['Position']
        ahead_row = same_lap[same_lap['Position'] == ahead_pos]
        if len(ahead_row) > 0:
            gap_ahead = ahead_row.iloc[0].get('GapBehind', np.nan)

    # Gap behind — GapBehind of the driver itself
    driver_same_lap = same_lap[same_lap['Driver'] == driver]
    gap_behind = np.nan
    if len(driver_same_lap) > 0:
        gap_behind = driver_same_lap.iloc[0].get('GapBehind', np.nan)

    return (
        round(float(gap_ahead),  3) if not pd.isna(gap_ahead)  else np.nan,
        round(float(gap_behind), 3) if not pd.isna(gap_behind) else np.nan,
    )


def classify_context(
    gap_ahead: float,
    gap_behind: float,
) -> str:
    """
    Describes the gap state only — makes no strategic implication.
    The prediction model learns circuit-specific strategy from
    the data, not from these labels.

    tight_ahead   — car ahead is a racing reference
    tight_behind  — car behind is a racing reference  
    tight_both    — racing references both sides
    free          — no immediate racing references
    """
    if pd.isna(gap_ahead) or pd.isna(gap_behind):
        return 'unknown'

    tight_ahead  = gap_ahead  < 5.0
    tight_behind = gap_behind < 5.0

    if tight_ahead and tight_behind:
        return 'tight_both'
    if tight_ahead:
        return 'tight_ahead'
    if tight_behind:
        return 'tight_behind'
    return 'free'
    
def find_threat_gap(
    lap_slice: pd.DataFrame,
    laps: pd.DataFrame,
    driver: str,
    driver_pos: float,
    driver_lap_num: float,
    lap_number: float,
    circuit: str,
    year: int,
    context: str,
    pit_window_laps: int = 3,
) -> tuple[float, str]:
    """
    Returns the gap to the first non-pitting car behind.

    This is the ONLY metric that determines whether a pit
    window is open. The car behind that stays out is what
    closes the window — they will pass the pitting driver
    during the stationary time and determine the rejoin gap.

    A window is open when:
        threat_gap > pit_loss + clean_air_buffer

    Context is recorded as a feature for the prediction model
    but does NOT change the scan direction. Whether a driver
    is in an undercut battle or running free, the window
    calculation is the same physical question.
    """
    def pits_soon(candidate_driver: str) -> bool:
        return len(laps[
            (laps['Driver']    == candidate_driver) &
            (laps['LapNumber'].between(lap_number, lap_number + pit_window_laps)) &
            (laps['Circuit']   == circuit) &
            (laps['Year']      == year) &
            (laps['PitOutTime'].notna())
        ]) > 0

    # Same-lap cars only — exclude lapping leaders
    same_lap = lap_slice[
        lap_slice['LapNumber'] == driver_lap_num
    ].sort_values('Position').reset_index(drop=True)

    driver_idx_list = same_lap[same_lap['Driver'] == driver].index.tolist()
    if not driver_idx_list:
        return np.nan, 'no_data'
    driver_idx = driver_idx_list[0]

    # Always scan behind — accumulate gap until first non-pitting car
    cumulative = 0.0
    for i in range(driver_idx, len(same_lap) - 1):
        gb = same_lap.iloc[i].get('GapBehind', np.nan)
        if pd.isna(gb):
            continue
        cumulative += gb
        candidate = same_lap.iloc[i + 1]['Driver']
        if not pits_soon(candidate):
            return round(cumulative, 3), f"behind:{candidate}"

    # All cars behind are pitting — window is effectively open
    return round(cumulative, 3) if cumulative > 0 else np.nan, 'behind:all_pitting'
    
# ─────────────────────────────────────────────
# Phase 2 check
# ─────────────────────────────────────────────

def check_context_classifier(year: int = 2023, round_number: int = 2) -> None:
    """
    Load one race and verify context classification and
    threat gap computation are producing sensible outputs.
    """
    _, all_laps = load_race_laps(year, round_number)
    if all_laps is None:
        return

    gaps_df = compute_inter_car_gaps(all_laps)
    all_laps = all_laps.merge(
        gaps_df[['Driver', 'LapNumber', 'Circuit', 'Year',
                 'GapAhead', 'GapBehind']],
        on=['Driver', 'LapNumber', 'Circuit', 'Year'],
        how='left'
    )

    circuit = all_laps['Circuit'].iloc[0]
    sample_lap = int(all_laps['LapNumber'].median())

    lap_slice = all_laps[
        all_laps['LapNumber'] == sample_lap
    ].sort_values('Position')

    print(f"\nContext check — {year} R{round_number} {circuit} lap {sample_lap}")
    print(f"  {'Driver':<6} {'Pos':>4} {'GapAhead':>10} {'GapBehind':>10} {'Context':<12} {'ThreatGap':>10} {'Threat'}")
    print(f"  {'-'*75}")

    for _, row in lap_slice.iterrows():
        driver = row['Driver']
        pos    = row.get('Position', np.nan)

        if pd.isna(pos):
            continue

        driver_lap_num = row['LapNumber']
        ga, gb = get_racing_gaps(lap_slice, driver)
        if pd.isna(ga) or pd.isna(gb):
            continue

        ctx = classify_context(ga, gb)

        threat_gap, threat_desc = find_threat_gap(
            lap_slice   = lap_slice,
            laps        = all_laps,
            driver      = driver,
            driver_pos  = pos,
            driver_lap_num = driver_lap_num,
            lap_number  = sample_lap,
            circuit     = circuit,
            year        = year,
            context     = ctx,
        )

        print(f"  {driver:<6} {int(pos):>4} {ga:>10.2f} {gb:>10.2f} "
              f"{ctx:<12} "
              f"{threat_gap:>10.2f} {threat_desc}")

### Phase 3 — Degradation model

In [ ]:
def fuel_correction(lap_number: float) -> float:
    """
    Lap time reduction due to fuel burn by this lap.
    Lap 1 baseline = 0s correction.
    Each subsequent lap the car is lighter by FUEL_BURN_RATE kg,
    gaining FUEL_LAP_EFFECT seconds of pace.

    Example: lap 30 → 29 laps burned → 29 × 1.8 × 0.035 = 1.827s
    faster than lap 1 due to fuel alone. Without this correction
    early laps look artificially degraded because the car is
    heavier, not because the tyre is worn.
    """
    return (lap_number - 1) * FUEL_BURN_RATE * FUEL_LAP_EFFECT


def compute_crossover_lap(
    coeffs_a: np.ndarray,
    coeffs_b: np.ndarray,
    max_laps: int = 60
) -> int | None:
    """
    Find the stint lap at which compound B cumulative time
    becomes lower than compound A cumulative time.

    This is the strategic crossover point — beyond this lap
    a driver on compound B has spent less total time than
    one who started on compound A, even though B is slower
    per lap early in the stint.
    """
    poly_a = np.poly1d(coeffs_a)
    poly_b = np.poly1d(coeffs_b)
    for lap in range(1, max_laps):
        laps_arr = np.arange(1, lap + 1)
        if np.sum(poly_b(laps_arr)) < np.sum(poly_a(laps_arr)):
            return lap
    return None


def fit_degradation_model(
    clean_laps: pd.DataFrame
) -> dict[str, dict[str, np.ndarray]]:
    """
    Fit a quadratic degradation curve per compound per circuit
    on fuel-corrected lap times.

    Returns nested dict:
        coeffs[circuit][compound] = np.ndarray of [a2, a1, a0]

    These coefficients are used at prediction time to estimate
    how much slower a tyre will be at a given age, which feeds
    into the crossover point calculation and stint length targets.
    """
    coeffs: dict[str, dict[str, np.ndarray]] = {}

    for circuit in clean_laps['Circuit'].unique():
        profile = CIRCUIT_PROFILES.get(circuit)
        if profile is None:
            continue

        coeffs[circuit] = {}
        circuit_laps = clean_laps[clean_laps['Circuit'] == circuit]

        for compound in ['SOFT', 'MEDIUM', 'HARD']:
            stint_laps = circuit_laps[
                (circuit_laps['Compound'] == compound) &
                (circuit_laps['TyreLife'] > 1)
            ].copy()

            if len(stint_laps) < 15:
                continue

            # Fuel-corrected pace delta from each driver's baseline
            stint_laps['FuelCorr']  = stint_laps['LapNumber'].apply(fuel_correction)
            stint_laps['CorrTime']  = stint_laps['LapTimeSeconds'] - stint_laps['FuelCorr']
            stint_laps['BasePace']  = stint_laps.groupby('DriverNumber')['CorrTime'].transform('min')
            stint_laps['PaceDelta'] = stint_laps['CorrTime'] - stint_laps['BasePace']

            # Remove outliers — delta > 5s is a slow lap, not degradation
            stint_laps = stint_laps[stint_laps['PaceDelta'].between(0, 5)]
            if len(stint_laps) < 10:
                continue

            try:
                c = np.polyfit(
                    stint_laps['TyreLife'].values.astype(float),
                    stint_laps['PaceDelta'].values,
                    deg=2
                )
                coeffs[circuit][compound] = c
            except np.linalg.LinAlgError:
                continue

    return coeffs


def evaluate_degradation_model(
    train_coeffs: dict[str, dict[str, np.ndarray]],
    test_laps: pd.DataFrame
) -> pd.DataFrame:
    """
    Measure RMSE and MAE of the fitted degradation curves
    against held-out test laps.

    Also computes compound crossover laps per circuit —
    the stint length beyond which a harder compound would
    have been faster in total.
    """
    results = []

    for circuit, compound_coeffs in train_coeffs.items():
        circuit_test = test_laps[test_laps['Circuit'] == circuit]

        for compound, coeffs in compound_coeffs.items():
            test = circuit_test[
                (circuit_test['Compound'] == compound) &
                (circuit_test['TyreLife'] > 1)
            ].copy()

            if len(test) < 5:
                continue

            test['FuelCorr']  = test['LapNumber'].apply(fuel_correction)
            test['CorrTime']  = test['LapTimeSeconds'] - test['FuelCorr']
            test['BasePace']  = test.groupby('DriverNumber')['CorrTime'].transform('min')
            test['PaceDelta'] = test['CorrTime'] - test['BasePace']
            test = test[test['PaceDelta'].between(0, 5)]

            if len(test) < 5:
                continue

            poly   = np.poly1d(coeffs)
            y_pred = poly(test['TyreLife'].values.astype(float))
            y_test = test['PaceDelta'].values

            rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
            mae  = float(mean_absolute_error(y_test, y_pred))

            results.append({
                'circuit':  circuit,
                'compound': compound,
                'rmse_s':   round(rmse, 4),
                'mae_s':    round(mae, 4),
                'n_test':   len(test),
            })

    df = pd.DataFrame(results)

    # Crossover points
    print("\n  Crossover laps (fuel-corrected):")
    for circuit, compound_coeffs in train_coeffs.items():
        pairs = [
            ('SOFT',   'MEDIUM'),
            ('SOFT',   'HARD'),
            ('MEDIUM', 'HARD'),
        ]
        for c_a, c_b in pairs:
            if c_a in compound_coeffs and c_b in compound_coeffs:
                lap = compute_crossover_lap(
                    compound_coeffs[c_a],
                    compound_coeffs[c_b]
                )
                print(f"    {circuit:<30} {c_a} → {c_b}: lap {lap}")

    return df


# ─────────────────────────────────────────────
# Phase 3 check
# ─────────────────────────────────────────────

def check_degradation_model(
    train_years: list[int] = [2020, 2021],
    test_years:  list[int] = [2023, 2024]
) -> None:
    print("\nLoading train data...")
    train_clean, _ = load_dataset(train_years)

    print("\nLoading test data...")
    test_clean, _  = load_dataset(test_years)

    print("\nFitting degradation model...")
    coeffs = fit_degradation_model(train_clean)
    print(f"  Fitted {sum(len(v) for v in coeffs.values())} compound curves "
          f"across {len(coeffs)} circuits")

    print("\nEvaluating on test data...")
    results = evaluate_degradation_model(coeffs, test_clean)

    if len(results) > 0:
        print(f"\n{results.to_string(index=False)}")
        print(f"\n  Mean RMSE: {results['rmse_s'].mean():.4f}s")
        print(f"  Mean MAE : {results['mae_s'].mean():.4f}s")
    else:
        print("  No test results — check circuit overlap between train and test years")

In [ ]:
# ─────────────────────────────────────────────
# Phase 4 — Forward-looking label construction
# ─────────────────────────────────────────────

# ─────────────────────────────────────────────
# Pace-adjusted window check
# ─────────────────────────────────────────────

def is_window_viable(
    threat_gap: float,
    pit_loss: float,
    own_pace: float,
    threat_pace: float,
    clean_air_buffer: float = CLEAN_AIR_BUFFER,
) -> bool:
    """
    Dynamic window viability check accounting for relative pace.

    A Red Bull 8s behind a Williams closes that gap in very
    few laps. A Williams 8s behind a Red Bull never closes it.
    The static threshold treats both identically — this does not.

    closing_rate = own_pace - threat_pace
      positive → threat car is faster, gap is closing
      negative → threat car is slower, gap is growing
      zero     → same pace, static gap

    Window is open if the threat car cannot close the gap
    during the pit stop plus the clean air buffer period.
    """
    closing_rate = own_pace - threat_pace

    if closing_rate <= 0:
        # Threat car same pace or slower — gap not closing
        # Use static check
        return threat_gap > pit_loss + clean_air_buffer

    # Threat car is faster — check if they close the gap
    # before driver rejoins with clean air
    # Time for threat to cover threat_gap at closing_rate laps/s
    laps_until_caught = threat_gap / closing_rate
    laps_lost         = pit_loss / own_pace

    return laps_until_caught > laps_lost + (clean_air_buffer / own_pace)


def get_recent_pace(
    laps: pd.DataFrame,
    driver: str,
    lap_number: float,
    circuit: str,
    year: int,
    window: int = 5,
) -> float:
    """
    Average lap time over the last N clean (green flag) laps.
    Returns nan if fewer than 2 clean laps available.
    """
    recent = laps[
        (laps['Driver']      == driver) &
        (laps['LapNumber'].between(lap_number - window, lap_number - 1)) &
        (laps['Circuit']     == circuit) &
        (laps['Year']        == year) &
        (laps['TrackStatus'] == '1') &
        (laps['LapTimeSeconds'].notna())
    ]['LapTimeSeconds']

    return float(recent.mean()) if len(recent) >= 2 else np.nan

def construct_forward_labels(
    laps: pd.DataFrame,
    train_coeffs: dict[str, dict[str, np.ndarray]],
    forecast_horizon: int = FORECAST_HORIZON,
    clean_air_buffer: float = CLEAN_AIR_BUFFER,
    min_lap: int = 5,
    max_lap_fraction: float = 0.85,
) -> pd.DataFrame:

    gaps_df = compute_inter_car_gaps(laps)
    laps = laps.merge(
        gaps_df[['Driver', 'LapNumber', 'Circuit', 'Year',
                 'GapAhead', 'GapBehind']],
        on=['Driver', 'LapNumber', 'Circuit', 'Year'],
        how='left'
    )

    total_laps_map = (
        laps.groupby(['Circuit', 'Year'])['LapNumber']
        .max()
        .to_dict()
    )

    # ── Precompute circuit/year level stats ───────────────
    # Done once here rather than inside the driver loop
    # so we are not recomputing for every driver every lap
    circuit_stats: dict[tuple, dict] = {}
    for (circuit, year), race_laps in laps.groupby(['Circuit', 'Year']):
        profile = CIRCUIT_PROFILES.get(circuit)
        if profile is None:
            continue

        pit_loss = profile['pit_loss']

        median_gap = float(
            race_laps['GapBehind'].dropna().median()
        )

        median_lap_time = float(
            race_laps[race_laps['LapTimeSeconds'] > 60]['LapTimeSeconds'].median()
        )

        pit_loss_fraction = (
            pit_loss / median_lap_time
            if median_lap_time > 0 else np.nan
        )
        # ── VSC/SC rates for this circuit ─────────────────
        sc_laps = race_laps['TrackStatus'].apply(
            lambda s: any(c in {'4', '5'} for c in str(s))
        ).sum()
        vsc_laps = race_laps['TrackStatus'].apply(
            lambda s: any(c in {'6', '7'} for c in str(s))
        ).sum()
        total_race_laps = len(race_laps)

        sc_rate  = round(sc_laps  / total_race_laps, 4) if total_race_laps > 0 else 0.0
        vsc_rate = round(vsc_laps / total_race_laps, 4) if total_race_laps > 0 else 0.0

        circuit_stats[(circuit, year)] = {
            'pit_loss':           pit_loss,
            'circuit_median_gap': round(median_gap, 3),
            'pit_loss_fraction':  round(pit_loss_fraction, 4),
            'sc_rate':            sc_rate,
            'vsc_rate':           vsc_rate,
        }
        
    records = []

    for (circuit, year, driver), driver_laps in laps.groupby(
        ['Circuit', 'Year', 'Driver']
    ):
        stats = circuit_stats.get((circuit, year))
        if stats is None:
            continue

        pit_loss          = stats['pit_loss']
        circuit_median_gap = stats['circuit_median_gap']
        pit_loss_fraction  = stats['pit_loss_fraction']
        sc_rate            = stats['sc_rate']       # ← add
        vsc_rate           = stats['vsc_rate']      # ← add
        threshold          = pit_loss + clean_air_buffer
        total              = total_laps_map.get((circuit, year), 999)

        driver_laps = driver_laps.sort_values('LapNumber').reset_index(drop=True)

        for i, row in driver_laps.iterrows():
            lap_num  = row['LapNumber']
            position = row.get('Position', np.nan)

            # Skip opening laps, closing laps, and pit lane laps
            if lap_num < min_lap:
                continue
            if lap_num > total * max_lap_fraction:
                continue
            if pd.notna(row.get('PitInTime')) or pd.notna(row.get('PitOutTime')):
                continue
            if pd.isna(position):
                continue

            # Current lap slice — all cars on this lap
            lap_slice = laps[
                (laps['LapNumber'] == lap_num) &
                (laps['Circuit']   == circuit) &
                (laps['Year']      == year)
            ].sort_values('Position')

            # Racing gaps — same-lap cars only
            ga, gb = get_racing_gaps(lap_slice, driver)
            if pd.isna(ga) or pd.isna(gb):
                continue

            context = classify_context(ga, gb)
            if context == 'unknown':
                continue

            driver_lap_num = row['LapNumber']

            # Current effective threat gap
            current_threat_gap, current_threat_desc = find_threat_gap(
                lap_slice      = lap_slice,
                laps           = laps,
                driver         = driver,
                driver_pos     = position,
                driver_lap_num = driver_lap_num,
                lap_number     = lap_num,
                circuit        = circuit,
                year           = year,
                context        = context,
            )

            if pd.isna(current_threat_gap):
                continue

            # ── Relative pace ─────────────────────────────────
            own_pace = get_recent_pace(
                laps, driver, lap_num, circuit, year
            )

            # Parse threat driver from description
            # format is 'behind:VER' or 'behind:all_pitting'
            threat_driver_name = (
                current_threat_desc.split(':')[1]
                if ':' in current_threat_desc and
                current_threat_desc.split(':')[1] != 'all_pitting'
                else None
            )

            threat_pace = (
                get_recent_pace(laps, threat_driver_name, lap_num, circuit, year)
                if threat_driver_name else np.nan
            )

            closing_rate = (
                round(own_pace - threat_pace, 4)
                if not pd.isna(own_pace) and not pd.isna(threat_pace)
                else 0.0
            )

            # ── Window viability ──────────────────────────────
            if not pd.isna(own_pace) and not pd.isna(threat_pace):
                gap_is_open_now = int(is_window_viable(
                    threat_gap       = current_threat_gap,
                    pit_loss         = pit_loss,
                    own_pace         = own_pace,
                    threat_pace      = threat_pace,
                    clean_air_buffer = clean_air_buffer,
                ))
            else:
                # Fallback to static check if pace unavailable
                gap_is_open_now = int(
                    current_threat_gap > pit_loss + clean_air_buffer
                )

            # ── Gap trend — last 3 laps ───────────────────────
            recent_gaps = []
            for past_offset in [3, 2, 1]:
                past_lap = lap_num - past_offset
                past_slice = laps[
                    (laps['LapNumber'] == past_lap) &
                    (laps['Circuit']   == circuit) &
                    (laps['Year']      == year)
                ].sort_values('Position')
                if len(past_slice) == 0:
                    continue
                past_ga, past_gb = get_racing_gaps(past_slice, driver)
                past_ctx = classify_context(
                    past_ga if not pd.isna(past_ga) else ga,
                    past_gb if not pd.isna(past_gb) else gb
                )
                past_threat, _ = find_threat_gap(
                    lap_slice      = past_slice,
                    laps           = laps,
                    driver         = driver,
                    driver_pos     = position,
                    driver_lap_num = past_lap,
                    lap_number     = past_lap,
                    circuit        = circuit,
                    year           = year,
                    context        = past_ctx,
                )
                if not pd.isna(past_threat):
                    recent_gaps.append(past_threat)

            gap_trend = 0.0
            if len(recent_gaps) >= 2:
                gap_trend = float(
                    np.polyfit(range(len(recent_gaps)), recent_gaps, 1)[0]
                )

            # ── Degradation delta at current tire age ─────────
            tire_age = int(row.get('TyreLife', 1))
            stint_number = int(row.get('Stint', 1))  
            compound = row.get('Compound', 'UNKNOWN')
            deg_delta = 0.0
            circuit_coeffs = train_coeffs.get(circuit, {})
            if compound in circuit_coeffs:
                poly      = np.poly1d(circuit_coeffs[compound])
                deg_delta = float(poly(tire_age))

            

            # ── Cars behind pitting soon ──────────────────────
            cars_behind_pitting = len(laps[
                (laps['LapNumber'].between(lap_num, lap_num + 3)) &
                (laps['Circuit']   == circuit) &
                (laps['Year']      == year) &
                (laps['PitOutTime'].notna())
            ])

            # ── Forward scan: find when gap first opens ────────
            laps_until_open = None
            window_open_lap = None

            for offset in range(forecast_horizon + 1):
                future_lap = lap_num + offset

                future_slice = laps[
                    (laps['LapNumber'] == future_lap) &
                    (laps['Circuit']   == circuit) &
                    (laps['Year']      == year)
                ].sort_values('Position')

                future_driver_row = future_slice[future_slice['Driver'] == driver]
                if len(future_driver_row) == 0:
                    break

                future_pos = future_driver_row.iloc[0].get('Position', np.nan)
                if pd.isna(future_pos):
                    break

                future_ga, future_gb = get_racing_gaps(future_slice, driver)
                if pd.isna(future_ga) or pd.isna(future_gb):
                    continue

                future_ctx = classify_context(future_ga, future_gb)

                future_threat, future_threat_desc = find_threat_gap(
                    lap_slice      = future_slice,
                    laps           = laps,
                    driver         = driver,
                    driver_pos     = future_pos,
                    driver_lap_num = future_lap,
                    lap_number     = future_lap,
                    circuit        = circuit,
                    year           = year,
                    context        = future_ctx,
                )

                if pd.isna(future_threat):
                    continue

                # ── Use pace-adjusted check in forward scan ────
                # Extract future threat driver for pace lookup
                future_threat_driver = (
                    future_threat_desc.split(':')[1]
                    if ':' in future_threat_desc and
                    future_threat_desc.split(':')[1] != 'all_pitting'
                    else None
                )

                future_own_pace = get_recent_pace(
                    laps, driver, future_lap, circuit, year
                )
                future_threat_pace = (
                    get_recent_pace(laps, future_threat_driver, future_lap, circuit, year)
                    if future_threat_driver else np.nan
                )

                if not pd.isna(future_own_pace) and not pd.isna(future_threat_pace):
                    window_open = is_window_viable(
                        threat_gap       = future_threat,
                        pit_loss         = pit_loss,
                        own_pace         = future_own_pace,
                        threat_pace      = future_threat_pace,
                        clean_air_buffer = clean_air_buffer,
                    )
                else:
                    # Fallback to static check
                    window_open = future_threat > threshold

                if window_open:
                    laps_until_open = offset
                    window_open_lap = int(future_lap)
                    break

            records.append({
                # identifiers unchanged
                'year':               year,
                'circuit':            circuit,
                'driver':             driver,
                'lap':                int(lap_num),
                # features
                'compound':           compound,
                'tire_age':           tire_age,
                'stint_number': stint_number,
                'context':            context,            # now tight_ahead/tight_behind/tight_both/free
                'gap_ahead':          round(ga, 3),
                'gap_behind':         round(gb, 3),
                'threat_gap':         round(current_threat_gap, 3),
                'gap_trend':          round(gap_trend, 4),
                'deg_delta':          round(deg_delta, 4),
                'cars_behind_pitting': cars_behind_pitting,
                'pit_loss':           pit_loss,
                'pit_loss_fraction':  round(pit_loss_fraction, 4) if not pd.isna(pit_loss_fraction) else np.nan,
                'circuit_median_gap': round(circuit_median_gap, 3),
                # targets unchanged
                'gap_is_open_now':    gap_is_open_now,
                'laps_until_open':    laps_until_open,
                'window_open_lap':    window_open_lap,
                'opens_in_horizon':   int(laps_until_open is not None),
                'own_pace':       round(own_pace,    3) if not pd.isna(own_pace)    else np.nan,
                'threat_pace':    round(threat_pace, 3) if not pd.isna(threat_pace) else np.nan,
                'closing_rate':   closing_rate,
                'race_progress': round(lap_num / total, 4),
                'sc_rate':            sc_rate,
                'vsc_rate':           vsc_rate,
            })

    return pd.DataFrame(records)


# ─────────────────────────────────────────────
# Phase 4 check
# ─────────────────────────────────────────────
def compute_vsc_sc_rates(
    laps: pd.DataFrame,
) -> dict[str, float]:
    """
    Compute historical VSC and SC deployment rates per circuit
    from the laps dataframe.

    Rate = fraction of laps run under VSC or SC conditions.
    Used as a feature representing the probability of a safety
    car period occurring at this circuit, which affects whether
    a team should stretch a stint to take a cheaper pit stop.

    TrackStatus codes:
        4  = Safety Car
        5  = Red Flag
        6  = VSC deployed
        7  = VSC ending
    """
    rates = {}
    sc_codes  = {'4', '5'}
    vsc_codes = {'6', '7'}

    for circuit, circuit_laps in laps.groupby('Circuit'):
        total = len(circuit_laps)
        if total == 0:
            continue

        sc_laps = circuit_laps['TrackStatus'].apply(
            lambda s: any(c in sc_codes for c in str(s))
        ).sum()

        vsc_laps = circuit_laps['TrackStatus'].apply(
            lambda s: any(c in vsc_codes for c in str(s))
        ).sum()

        rates[circuit] = {
            'sc_rate':  round(sc_laps  / total, 4),
            'vsc_rate': round(vsc_laps / total, 4),
        }

    return rates
    
def check_forward_labels(
    year: int = 2023,
    round_number: int = 2,
    train_years: list[int] = [2020, 2021]
) -> None:
    """
    Run forward label construction on one race and verify
    the label distribution looks physically sensible.
    """
    print(f"\nLoading train data for deg model...")
    train_clean, _ = load_dataset(train_years)
    train_coeffs   = fit_degradation_model(train_clean)

    print(f"\nLoading {year} R{round_number} for label construction...")
    _, all_laps = load_race_laps(year, round_number)
    if all_laps is None:
        return

    print("Building forward-looking labels...")
    labels = construct_forward_labels(all_laps, train_coeffs)

    if len(labels) == 0:
        print("No labels produced — check pipeline")
        return

    print(f"\nLabel summary — {year} R{round_number} "
          f"{all_laps['Circuit'].iloc[0]}")
    print(f"  Total lap-driver observations : {len(labels)}")
    print(f"  Gap open right now            : "
          f"{labels['gap_is_open_now'].mean():.1%}")
    print(f"  Opens within horizon          : "
          f"{labels['opens_in_horizon'].mean():.1%}")

    open_laps = labels[labels['laps_until_open'].notna()]['laps_until_open']
    print(f"\n  laps_until_open distribution:")
    print(f"    Median : {open_laps.median():.1f} laps")
    print(f"    Mean   : {open_laps.mean():.1f} laps")
    print(f"    p25    : {open_laps.quantile(0.25):.1f} laps")
    print(f"    p75    : {open_laps.quantile(0.75):.1f} laps")
    print(f"    = 0    : {(open_laps == 0).mean():.1%} (gap open immediately)")
    print(f"    1–5    : {((open_laps >= 1) & (open_laps <= 5)).mean():.1%}")
    print(f"    6–10   : {((open_laps >= 6) & (open_laps <= 10)).mean():.1%}")
    print(f"    11–20  : {((open_laps >= 11) & (open_laps <= 20)).mean():.1%}")

    print(f"\n  Context distribution:")
    for ctx, grp in labels.groupby('context'):
        print(f"    {ctx:<12} n={len(grp):>5}  "
              f"open_now={grp['gap_is_open_now'].mean():.1%}  "
              f"opens_in_horizon={grp['opens_in_horizon'].mean():.1%}  "
              f"mean_laps_until={grp['laps_until_open'].mean():.1f}")

    print(labels[[
        'driver', 'lap', 'context', 'threat_gap',
        'closing_rate', 'gap_is_open_now', 'laps_until_open'
    ]].head(10).to_string(index=False))

    print(f"\n  Pace data coverage:")
    print(f"    own_pace available   : {labels['own_pace'].notna().mean():.1%}")
    print(f"    threat_pace available: {labels['threat_pace'].notna().mean():.1%}")
    print(f"    closing_rate > 0     : {(labels['closing_rate'] > 0).mean():.1%}  "
          f"(threat car faster)")
    print(f"    closing_rate < 0     : {(labels['closing_rate'] < 0).mean():.1%}  "
          f"(threat car slower)")
    print(f"    mean closing_rate    : {labels['closing_rate'].mean():.3f}s/lap")


# ─────────────────────────────────────────────
# Phase 5 — Full dataset construction
# ─────────────────────────────────────────────

def build_full_dataset(
    train_years: list[int],
    label_years: list[int],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build the complete feature and label dataset for model training.

    train_years  — years used to fit the degradation model only.
                   These races do NOT appear in the label dataset
                   to prevent data leakage.

    label_years  — years used to construct forward-looking labels.
                   These are the observations the model trains on.

    Returns (labels_df, coeffs_dict) where:
        labels_df   — one row per driver per lap with features and targets
        coeffs_dict — fitted degradation coefficients (for feature use)
    """
    print("=" * 55)
    print("Phase 5 — Building full dataset")
    print("=" * 55)

    # Fit degradation model on training years
    print(f"\nStep 1 — Fitting degradation model on {train_years}...")
    train_clean, _ = load_dataset(train_years)
    train_coeffs   = fit_degradation_model(train_clean)
    print(f"  Fitted {sum(len(v) for v in train_coeffs.values())} "
          f"compound curves across {len(train_coeffs)} circuits")

    # Build forward labels on label years
    print(f"\nStep 2 — Building forward labels on {label_years}...")
    all_label_frames = []

    for year in label_years:
        rounds = get_rounds_for_circuits(year, list(CIRCUIT_PROFILES.keys()))
        print(f"\n  {year}: {len(rounds)} rounds")

        for rnd in rounds:
            print(f"    R{rnd}...", end=' ', flush=True)
            _, all_laps = load_race_laps(year, rnd)
            if all_laps is None or len(all_laps) == 0:
                print("skipped")
                continue

            try:
                labels = construct_forward_labels(all_laps, train_coeffs)
                if len(labels) > 0:
                    all_label_frames.append(labels)
                    print(f"{len(labels)} observations")
                else:
                    print("0 observations")
            except Exception as e:
                import traceback
                print(f"error: {e}")
                traceback.print_exc()
                break   # stop after first error so we can see it clearly

    if not all_label_frames:
        raise ValueError("No label data produced")

    full_df = pd.concat(all_label_frames, ignore_index=True)

    print(f"\n{'='*55}")
    print(f"Dataset summary")
    print(f"{'='*55}")
    print(f"  Total observations    : {len(full_df):,}")
    print(f"  Circuits              : {full_df['circuit'].nunique()}")
    print(f"  Drivers               : {full_df['driver'].nunique()}")
    print(f"  Years                 : {sorted(full_df['year'].unique())}")
    print(f"  Gap open now          : {full_df['gap_is_open_now'].mean():.1%}")
    print(f"  Opens in horizon      : {full_df['opens_in_horizon'].mean():.1%}")
    print(f"\n  laps_until_open (where window opens):")
    open_rows = full_df[full_df['laps_until_open'].notna()]['laps_until_open']
    print(f"    Median : {open_rows.median():.1f} laps")
    print(f"    Mean   : {open_rows.mean():.1f} laps")
    print(f"    p75    : {open_rows.quantile(0.75):.1f} laps")
    print(f"\n  Context distribution:")
    for ctx, grp in full_df.groupby('context'):
        print(f"    {ctx:<12} n={len(grp):>6,}  "
              f"open_now={grp['gap_is_open_now'].mean():.1%}  "
              f"opens_in_horizon={grp['opens_in_horizon'].mean():.1%}")
    print(f"\n  Pace coverage:")
    print(f"    own_pace    : {full_df['own_pace'].notna().mean():.1%}")
    print(f"    threat_pace : {full_df['threat_pace'].notna().mean():.1%}")

    return full_df, train_coeffs

# ─────────────────────────────────────────────
# Phase 6 — Model training and evaluation
# ─────────────────────────────────────────────

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

FEATURES = [
    'threat_gap',
    'closing_rate',
    'gap_ahead',
    'gap_behind',
    'gap_trend',
    'tire_age',
    'stint_number',
    'deg_delta',
    'cars_behind_pitting',
    'pit_loss',
    'pit_loss_fraction',
    'circuit_median_gap',
    'own_pace',
    'threat_pace',
    'race_progress',
    'sc_rate',
    'vsc_rate',       
]

CATEGORICAL_FEATURES = ['context', 'compound']


def prepare_features(
    df: pd.DataFrame,
    label_encoders: dict | None = None,
    fit_encoders: bool = False,
) -> tuple[pd.DataFrame, dict]:
    """
    Prepare feature matrix for model training or inference.

    Handles:
    - Label encoding of categorical features
    - Filling missing pace values with circuit median
    - Dropping rows with missing target or key features

    Returns (X, label_encoders) where X is the feature matrix
    and label_encoders can be reused at inference time.
    """
    df = df.copy()

    # Fill missing pace with circuit median pace
    # Missing pace = early laps before 5-lap window is available
    for col in ['own_pace', 'threat_pace']:
        circuit_medians = df.groupby('circuit')[col].transform('median')
        df[col] = df[col].fillna(circuit_medians)

    # Encode categorical features
    if label_encoders is None:
        label_encoders = {}

    for col in CATEGORICAL_FEATURES:
        if fit_encoders:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le
        else:
            le = label_encoders[col]
            # Handle unseen categories at inference time
            df[col] = df[col].astype(str).map(
                lambda x: le.transform([x])[0]
                if x in le.classes_ else -1
            )

    all_features = FEATURES + CATEGORICAL_FEATURES
    return df[all_features], label_encoders


def train_model(
    labels_path: str = './f1_pit_window_labels.csv',
    test_year: int = 2024,
    save_path: str = './f1_pit_window_model.pkl',
) -> None:
    """
    Train a gradient boosted regressor to predict laps_until_open.

    Split strategy:
        train — 2022 and 2023 data
        test  — 2024 data (held out, never seen during training)

    This is a temporal split — the model is always tested on
    data from after its training period, which mirrors real
    deployment where we predict future races from past data.

    Target: laps_until_open
        Rows where window never opens in horizon (laps_until_open=None)
        are set to forecast_horizon + 1 = 21, meaning "window does
        not open soon." This keeps them in the training set rather
        than discarding them — they teach the model what a closed
        window looks like.
    """
    print("=" * 55)
    print("Phase 6 — Model training")
    print("=" * 55)

    # Load dataset
    df = pd.read_csv(labels_path)
    print(f"\n  Loaded {len(df):,} observations")
    print(f"  Years available: {sorted(df['year'].unique())}")

    # Fill missing laps_until_open with horizon + 1
    HORIZON_SENTINEL = FORECAST_HORIZON + 1
    df['laps_until_open'] = df['laps_until_open'].fillna(HORIZON_SENTINEL)

    # Temporal train/test split
    train_df = df[df['year'] < test_year].copy()
    test_df  = df[df['year'] == test_year].copy()

    print(f"\n  Train: {len(train_df):,} rows "
          f"({sorted(train_df['year'].unique())})")
    print(f"  Test : {len(test_df):,} rows "
          f"({sorted(test_df['year'].unique())})")

    # Prepare features
    X_train, label_encoders = prepare_features(
        train_df, fit_encoders=True
    )
    X_test, _ = prepare_features(
        test_df, label_encoders=label_encoders, fit_encoders=False
    )

    y_train = train_df['laps_until_open'].values
    y_test  = test_df['laps_until_open'].values

    # Train gradient boosted regressor
    print("\n  Training GradientBoostingRegressor...")
    model = GradientBoostingRegressor(
        n_estimators      = 300,
        learning_rate     = 0.05,
        max_depth         = 5,
        min_samples_leaf  = 20,
        subsample         = 0.8,
        random_state      = 42,
        verbose           = 0,
    )
    model.fit(X_train, y_train)

    # Evaluate on held-out 2024 data
    y_pred = model.predict(X_test)
    y_pred = np.clip(y_pred, 0, HORIZON_SENTINEL)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Accuracy within N laps — more interpretable than MAE
    within_2  = np.mean(np.abs(y_pred - y_test) <= 2)
    within_5  = np.mean(np.abs(y_pred - y_test) <= 5)

    print(f"\n{'='*55}")
    print(f"Test results — {test_year} holdout")
    print(f"{'='*55}")
    print(f"  MAE              : {mae:.2f} laps")
    print(f"  RMSE             : {rmse:.2f} laps")
    print(f"  Within 2 laps    : {within_2:.1%}")
    print(f"  Within 5 laps    : {within_5:.1%}")

    # Per-context breakdown
    test_df = test_df.copy()
    test_df['predicted'] = y_pred
    test_df['error']     = np.abs(y_pred - y_test)

    print(f"\n  Per-context MAE:")
    for ctx, grp in test_df.groupby('context'):
        ctx_mae = grp['error'].mean()
        print(f"    {ctx:<12} MAE={ctx_mae:.2f} laps  "
              f"n={len(grp):,}")

    # Feature importances
    importances = pd.Series(
        model.feature_importances_,
        index=FEATURES + CATEGORICAL_FEATURES
    ).sort_values(ascending=False)

    print(f"\n  Feature importances (top 10):")
    for feat, imp in importances.head(10).items():
        bar = '█' * int(imp * 40)
        print(f"    {feat:<25} {imp:.4f}  {bar}")

    # Save model and encoders
    joblib.dump({'model': model, 'encoders': label_encoders}, save_path)
    print(f"\n  Model saved to {save_path}")

    return model, label_encoders, test_df

def tune_model(
    labels_path: str = './f1_pit_window_labels.csv',
    test_year:   int = 2024,
    save_path:   str = './f1_pit_window_model_tuned.pkl',
) -> None:
    """
    Grid search over key hyperparameters to minimise MAE
    on a validation split carved from the training data.

    Uses 2022 as validation, trains on 2023 only during search,
    then refits the best params on the full train set (2022+2023)
    and evaluates on the 2024 holdout.

    This avoids using 2024 test data to select hyperparameters,
    which would leak future information into the model selection.
    """
    from sklearn.model_selection import GridSearchCV, cross_val_score

    print("=" * 55)
    print("Phase 6b — Hyperparameter tuning")
    print("=" * 55)

    df = pd.read_csv(labels_path)
    HORIZON_SENTINEL = FORECAST_HORIZON + 1
    df['laps_until_open'] = df['laps_until_open'].fillna(HORIZON_SENTINEL)

    train_df = df[df['year'] < test_year].copy()
    test_df  = df[df['year'] == test_year].copy()

    X_train, label_encoders = prepare_features(train_df, fit_encoders=True)
    X_test, _               = prepare_features(test_df, label_encoders=label_encoders)

    y_train = train_df['laps_until_open'].values
    y_test  = test_df['laps_until_open'].values

    # ── Grid search ───────────────────────────────────────
    param_grid = {
        'n_estimators':     [200, 300, 500],
        'max_depth':        [4, 5, 6],
        'learning_rate':    [0.03, 0.05, 0.1],
        'min_samples_leaf': [10, 20, 30],
    }

    base_model = GradientBoostingRegressor(
        subsample    = 0.8,
        random_state = 42,
    )

    print("\n  Running grid search (this takes 5–15 minutes)...")
    print(f"  Parameter combinations: "
          f"{len(param_grid['n_estimators']) * len(param_grid['max_depth']) * len(param_grid['learning_rate']) * len(param_grid['min_samples_leaf'])}")

    grid_search = GridSearchCV(
        estimator  = base_model,
        param_grid = param_grid,
        scoring    = 'neg_mean_absolute_error',
        cv         = 3,
        n_jobs     = -1,       # use all CPU cores
        verbose    = 1,
    )
    grid_search.fit(X_train, y_train)

    best_params = grid_search.best_params_
    best_cv_mae = -grid_search.best_score_

    print(f"\n  Best params   : {best_params}")
    print(f"  Best CV MAE   : {best_cv_mae:.3f} laps")

    # ── Refit on full train set with best params ──────────
    print("\n  Refitting on full train set with best params...")
    best_model = GradientBoostingRegressor(
        subsample    = 0.8,
        random_state = 42,
        **best_params,
    )
    best_model.fit(X_train, y_train)

    # ── Evaluate on 2024 holdout ──────────────────────────
    y_pred = np.clip(best_model.predict(X_test), 0, HORIZON_SENTINEL)

    mae     = mean_absolute_error(y_test, y_pred)
    rmse    = np.sqrt(mean_squared_error(y_test, y_pred))
    within2 = np.mean(np.abs(y_pred - y_test) <= 2)
    within5 = np.mean(np.abs(y_pred - y_test) <= 5)

    print(f"\n{'='*55}")
    print(f"Tuned model — 2024 holdout results")
    print(f"{'='*55}")
    print(f"  MAE              : {mae:.2f} laps  (baseline: 2.89)")
    print(f"  RMSE             : {rmse:.2f} laps  (baseline: 4.59)")
    print(f"  Within 2 laps    : {within2:.1%}  (baseline: 54.4%)")
    print(f"  Within 5 laps    : {within5:.1%}  (baseline: 79.1%)")

    # ── Per-context breakdown ─────────────────────────────
    test_df = test_df.copy()
    test_df['predicted'] = y_pred
    test_df['error']     = np.abs(y_pred - y_test)

    print(f"\n  Per-context MAE (tuned):")
    for ctx, grp in test_df.groupby('context'):
        ctx_mae     = grp['error'].mean()
        ctx_mae_old = {'free': 2.85, 'tight_ahead': 3.22,
                       'tight_behind': 2.17, 'tight_both': 2.98}.get(ctx, 0)
        delta = ctx_mae - ctx_mae_old
        print(f"    {ctx:<12} MAE={ctx_mae:.2f}  "
              f"delta={delta:+.2f}  n={len(grp):,}")

    # ── Feature importances ───────────────────────────────
    importances = pd.Series(
        best_model.feature_importances_,
        index=FEATURES + CATEGORICAL_FEATURES
    ).sort_values(ascending=False)

    print(f"\n  Feature importances (tuned):")
    for feat, imp in importances.head(10).items():
        bar = '█' * int(imp * 40)
        print(f"    {feat:<25} {imp:.4f}  {bar}")

    # ── Save ─────────────────────────────────────────────
    joblib.dump({
        'model':    best_model,
        'encoders': label_encoders,
        'params':   best_params,
    }, save_path)
    print(f"\n  Tuned model saved to {save_path}")

    return best_model, label_encoders, test_df

#### Runs all the code could take multiple minutes.

In [ ]:
if __name__ == '__main__':
    # ── Load pre-built dataset ────────────────────────────
    # Assumes f1_pit_window_labels.csv is in the working directory
    import pandas as pd
    full_df = pd.read_csv('./f1_pit_window_labels.csv')
    print(f"Loaded dataset: {len(full_df):,} observations")

    # ── Load pre-trained model ────────────────────────────
    # Assumes f1_pit_window_model.pkl is in the working directory
    import joblib
    artefacts     = joblib.load('./f1_pit_window_model.pkl')
    model         = artefacts['model']
    label_encoders = artefacts['encoders']
    print(f"Loaded model: {type(model).__name__}")

    # ── Fit degradation coefficients ──────────────────────
    # Needed for construct_forward_labels at inference time
    # Uses cached FastF1 data — no new downloads required
    print("\nFitting degradation model from cached data...")
    train_clean, _ = load_dataset([2020, 2021])
    train_coeffs   = fit_degradation_model(train_clean)
    print(f"  {sum(len(v) for v in train_coeffs.values())} compound curves fitted")

    # ── Quick validation on 2024 holdout ─────────────────
    print("\nRunning validation on 2024 holdout...")
    model, encoders, test_results = train_model(
        labels_path = './f1_pit_window_labels.csv',
        test_year   = 2024,
    )

### Expected Analytical Outcomes

**1. Model accuracy (primary metric):**
We expect the tuned GBM to achieve a **mean absolute error of approximately 2.5–3.5 laps** on the 2024 holdout, with roughly **50–60% of predictions within 2 laps** of the true window opening and **75–85% within 5 laps**. These targets are grounded in the physical predictability ceiling of the problem — late-race safety cars and sudden pace swings introduce irreducible uncertainty that no model can eliminate.

**2. Feature importance ranking:**
We expect `threat_gap` to be the dominant feature (it most directly determines window viability), followed by `closing_rate` (pace differential drives how quickly the gap changes), `tire_age` and `deg_delta` (higher tire age increases strategic urgency), and `race_progress` (end-of-race laps have narrowing pit options). The `sc_rate` and `vsc_rate` features are expected to provide modest but meaningful signal, particularly at circuits like Monaco and Singapore where safety cars are historically frequent.

**3. Per-context performance differences:**
We expect the model to perform best in the `free` context (no immediate racing references, gap dynamics are smooth and predictable) and worst in `tight_both` (competing pressures from both ahead and behind create abrupt gap changes that are harder to forecast). The `tight_behind` context — the classic undercut threat scenario — is expected to be the most strategically valuable to predict accurately, even if slightly harder than `free`.

**4. Compound crossover laps:**
From the degradation model, we expect to observe crossover laps in the range of **12–18 laps** for SOFT→MEDIUM and **20–30 laps** for MEDIUM→HARD at most circuits, consistent with publicly documented F1 strategy patterns. High-degradation circuits (Bahrain, Abu Dhabi) will show shorter crossover laps; low-degradation circuits (Monaco, Hungary) will show longer ones.

**5. 2024 vs. 2025 predictability comparison:**
From the race position classifier, we expect 2025 accuracy curves to be slightly lower and slower-rising than 2024, reflecting a more competitive and unpredictable field as teams converge in car performance under the current regulations in their third year. If 2025 instead shows higher early-race accuracy, it would suggest continued dominance by one or two teams — a meaningful sporting insight beyond the purely technical modeling result.

## Modeling the Effects of Temperature on Race Performance

This will seek to work with the same set as the prediction modeling for driver behavior, but uses the weather dataset for session data in the analysis.

In [ ]:
races = ['Bahrain', 'Saudi Arabia', 'Australia', 'Japan', 'China']
years = [2024, 2025]
all_data = []

for race in races:
    for year in years: 
        session = fastf1.get_session(year, race, 'R')
        session.load(telemetry=False, weather=True)
        
        laps = session.laps.copy()
        weather = session.weather_data.copy()
        
        laps['Race'] = race
        laps['Year'] = year
        
        laps = laps.sort_values('Time')
        weather = weather.sort_values('Time')
    
        laps_with_weather = pd.merge_asof(laps, weather, on='Time', direction='backward')
        
        all_data.append(laps_with_weather)

combined_df = pd.concat(all_data, ignore_index=True)

combined_df['LapTime_Seconds'] = final_df['LapTime'].dt.total_seconds()

print(combined_df[['Race', 'LapNumber', 'LapTime_Seconds', 'AirTemp']].head())

In [ ]:
import statsmodels.formula.api as smf

# Multi-variate regression
reg = smf.ols(
    formula = '''LapTime_Seconds ~ 
                + C(Year)
                + C(Race)
                + C(Compound) - 1
                + TrackTemp
                + AirTemp
                + Pressure
                + Humidity
                + WindSpeed
                + WindDirection
                ''',
              data = combined_df
          ).fit(cov_type = 'HC2')

print(reg.summary())

# Temperature Scatter Plot
fig, ax = plt.subplots(figsize=(10, 5))

for race, group in combined_df[combined_df['Year'] == 2024].groupby('Race'):
    ax.scatter(group['TrackTemp'], group['LapTime_Seconds'], label=race, alpha=0.5)

ax.set_title('2024 Races by Country and Temperature')
ax.set_xlabel('Track Temperature')
ax.set_ylabel('Lap Time (seconds)')

ax.legend()
plt.tight_layout()
plt.show()

## Analysis

**Temperature**: The outcomes of the regression more or less align with the hypothesis that track temperature would have a statistically signficant effect on the race time of the drivers. 

**Tire Compound**: each tire compound was statistically signficant with the following effects:
- **Soft**: showed a positive coefficient, which increased lap times
- **Medium**: showed a negative coefficient, which decreased lap times the most
- **Intermediate** had a weak positive coefficient

**Non-statistically Significant Factors**: the only variables that showed lack of statistical signficance at the 95% level was AirTemp and WindDirection

## 6. Peer Feedback

**Feedback group:** Joely Marsyla, Caitlin Roake, Natalie Zorn

---

Our peer reviewers provided thoughtful and constructive feedback during the in-class milestone presentation. Below is a summary of the key points raised and our planned responses.

### 6.1 Feedback Received

**Strength — Strong domain grounding and clear motivation**
> The group noted that the project was exceptionally well-motivated, with a clear real-world connection to how actual F1 strategy teams operate. The inclusion of domain-specific details (e.g., the fuel correction factor, circuit-specific pit loss times) demonstrated genuine investment in the subject matter and made the proposal feel credible beyond a generic sports analytics project.

**Concern — Model evaluation / train-test split clarity**
> Reviewers pointed out that the preliminary race position classifier (Section 5.4) trained and evaluated on the same dataset, which risks overfitting and makes the reported accuracy figures hard to interpret. They recommended implementing a proper temporal holdout — for example, training on 2022–2023 races and testing exclusively on 2024 — before drawing conclusions about model quality.

**Suggestion — Simplify the scope to ensure completion**
> With five distinct research questions listed, the group expressed concern that the project may be too ambitious for the given timeline. They suggested prioritizing two or three core analyses (specifically the tire degradation model and pit window forecaster, as these are most novel and interconnected) and treating the race position classifier as a stretch goal rather than a primary deliverable.

**Question — How are lapped cars handled in gap calculations?**
> Natalie raised a specific technical question: when computing `GapAhead` and `GapBehind` using lap crossing times, how do lapped cars affect the gap computation? She noted that a driver who has lapped another car will appear artificially close in position-sorted order, inflating the apparent gap to the car "ahead." This is a legitimate concern that we had already addressed in the codebase (see the `same_lap` filter in the gap computation functions), and we clarified this in our response.

---

### 6.2 Our Response and Planned Adjustments

| Feedback | Action Taken / Planned |
|----------|----------------------|
| Overfitting risk in position classifier | Implementing temporal holdout: train on 2022–2023, test on 2024. Already done for pit window model. |
| Scope concerns | Designating tire degradation + pit window forecasting as primary deliverables; race position classifier as secondary |
| Lapped car gap contamination | Already handled via `same_lap` filter in `compute_inter_car_gaps()` — added clarifying comments and documentation |
| Model interpretability | Will add SHAP feature importance plots alongside built-in GBM importances for the final report |

## 7. Completed Milestones

The following project milestones have been completed as of this submission:

- [x] **Project scoping and research question finalization** — Five primary research questions defined; scope narrowed based on peer feedback to prioritize pit window forecasting and tire degradation as primary deliverables.
- [x] **Data acquisition pipeline** — FastF1 API integration complete; local caching infrastructure set up; all 2022–2024 race sessions verified as loadable.
- [x] **Base data quality filtering** — Lap time bounds filter, pit lap exclusion, and out-lap exclusion implemented and tested.
- [x] **Track status filter** — `is_clean_lap()` function correctly identifies and excludes SC, VSC, red flag, and yellow flag laps using composite `TrackStatus` codes.
- [x] **Inter-car gap computation** — `compute_inter_car_gaps()` implemented using lap crossing times, with same-lap filter to correctly exclude lapped cars.
- [x] **Context classification** — `classify_context()` categorizes each driver's gap state (tight_ahead / tight_behind / tight_both / free) per lap.
- [x] **Threat gap computation** — `find_threat_gap()` identifies the first non-pitting car behind and accumulates the cumulative gap, accounting for cars that will also pit in the next 3 laps.
- [x] **Fuel correction model** — Quadratic fuel weight correction implemented to separate fuel load effects from tire degradation in lap time analysis.
- [x] **Tire degradation model** — `fit_degradation_model()` fits compound-specific quadratic curves per circuit on fuel-corrected pace deltas; evaluated with RMSE/MAE on held-out data.
- [x] **Pace-adjusted window viability check** — `is_window_viable()` accounts for relative pace between driver and threat car rather than using a static gap threshold.
- [x] **Forward-looking label construction** — `construct_forward_labels()` produces `laps_until_open` targets with a 20-lap forecast horizon; pipeline tested on single-race validation.
- [x] **Preliminary race position classifier** — Random Forest trained on 2024 data; predictability curve generated at lap checkpoints 5, 10, 20, 30, 40, 50.
- [x] **Gradient Boosted Regressor (pit window model)** — Full model training and evaluation pipeline implemented with temporal 2024 holdout; hyperparameter tuning via grid search scaffolded.
- [x] **Pipeline check functions** — `check_pipeline()`, `check_context_classifier()`, `check_degradation_model()`, and `check_forward_labels()` all passing on 2023 data.
- [x] **Peer feedback session** — Feedback received from Joely Marsyla, Caitlin Roake, and Natalie Zorn; key adjustments planned.

## 8. Remaining Methods Milestones

The following modeling and analysis tasks remain to be completed before the final submission:

- [ ] **Full dataset construction (2022–2024)** — Run `build_full_dataset()` across all three label years; serialize to `f1_pit_window_labels.csv` (~145K observations expected).
- [ ] **Temporal holdout for position classifier** — Retrain Random Forest on 2022–2023 races and evaluate on 2024 to produce honest accuracy estimates; compare 2024 vs 2025 predictability curves.
- [ ] **Hyperparameter tuning (pit window model)** — Execute `tune_model()` grid search (81 parameter combinations, ~5–15 min runtime); document best params and improvement over baseline.
- [ ] **Degradation curve visualization** — Plot fitted quadratic curves for SOFT/MEDIUM/HARD per circuit category (street circuit, high-speed, technical); compute and annotate compound crossover laps.
- [ ] **Temperature effect regression** — Linear regression of track temperature on fuel-corrected pace delta, controlling for compound and tire age; report coefficient and 95% CI per compound.
- [ ] **Per-context model evaluation** — Disaggregate pit window MAE by gap context (tight_ahead / free / etc.); investigate whether model underperforms in specific racing situations.
- [ ] **SHAP feature importance** — Add SHAP explainability plots to the GBM pit window model to complement built-in feature importances.
- [ ] **Circuit clustering** — K-Means on circuit-level strategy fingerprints (average tire life, pit frequency, SC rate, degradation slope); visualize with PCA scatter plot.
- [ ] **2025 live data integration** — Extend position classifier and pit window dataset with available 2025 races as they become available during the project period.
- [ ] **Final write-up and narrative** — Complete Results and Discussion sections; produce publication-quality figures; compile all notebooks into a coherent final submission.

## 9. Summary

This milestone report documents significant progress on our F1 Race Strategy Prediction project. We have built and validated a robust data engineering pipeline using the FastF1 API that correctly handles the subtleties of Formula 1 timing data — Safety Car lap exclusion, lapped car filtering in gap computation, fuel weight correction for tire degradation, and pace-adjusted pit window viability — all grounded in the physical realities of motorsport strategy.

**What we have accomplished:** The full data loading, feature engineering, and label construction pipeline is functional and validated on real 2023–2024 race data. A preliminary race position classifier (Random Forest) has been trained and produces a sensible predictability curve, showing accuracy increasing from ~40–50% at lap 5 to over 80% by lap 40 as race strategy dispersion resolves. The core pit window forecasting model (Gradient Boosted Regressor with 17 features) has been architected, and its training and evaluation scaffolding is complete.

**What remains:** The primary remaining tasks are running the full multi-year dataset construction (computationally intensive due to FastF1 API calls), executing the hyperparameter grid search for the pit window model, and producing the final suite of visualizations — degradation curves, compound crossover analysis, and circuit clustering. The 2025 season is ongoing, which presents an opportunity to extend the analysis with fresh data.

**Revised scope (per peer feedback):** We have narrowed our primary deliverables to the **tire degradation model** and **pit window forecasting model**, which are the most technically novel and tightly interconnected components. The race position classifier will be included as a secondary analysis. This focuses our remaining effort on producing two high-quality, well-evaluated models rather than five partially completed ones.

The project is on track for a complete final submission and represents a genuine contribution to publicly available F1 strategy analytics tooling.

In [21]:
import zipfile
import os
from pathlib import Path

def zip_working_directory(
    output_name: str = 'f1_pit_window_project.zip',
    exclude_dirs: list[str] = ['__pycache__', '.ipynb_checkpoints','F!p2.ipynb'],
    exclude_extensions: list[str] = ['.pyc'],
) -> None:
    """
    Zip the current working directory into a single archive.

    Excludes:
    - f1_cache   (FastF1 cache — large and rebuildable)
    - __pycache__ and .ipynb_checkpoints (generated files)
    - .pyc files (compiled bytecode)
    """
    cwd        = Path.cwd()
    output_path = cwd / output_name

    print(f"Zipping: {cwd}")
    print(f"Output : {output_path}")

    file_count = 0
    skip_count = 0

    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for file in cwd.rglob('*'):

            # Skip the output zip itself
            if file == output_path:
                continue

            # Skip excluded directories
            if any(excl in file.parts for excl in exclude_dirs):
                skip_count += 1
                continue

            # Skip excluded extensions
            if file.suffix in exclude_extensions:
                skip_count += 1
                continue

            # Only zip files, not directory entries
            if file.is_file():
                arcname = file.relative_to(cwd)
                zf.write(file, arcname)
                file_count += 1

    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"\n  Files zipped : {file_count}")
    print(f"  Files skipped: {skip_count}")
    print(f"  Archive size : {size_mb:.1f} MB")
    print(f"  Saved to     : {output_path}")


zip_working_directory()

Zipping: /Users/cooperkerr/2026-datascience-project
Output : /Users/cooperkerr/2026-datascience-project/f1_pit_window_project.zip

  Files zipped : 769
  Files skipped: 5
  Archive size : 125.3 MB
  Saved to     : /Users/cooperkerr/2026-datascience-project/f1_pit_window_project.zip
